In [0]:
# CELL 1 — Compute severity score for every road section
# Formula based on the Abed et al. research paper indicators

spark.sql("USE pothole_heatmap")

scored_df = spark.sql("""
    SELECT
        section_id,
        city,
        latitude,
        longitude,
        road_type,
        daily_traffic,
        accidents_per_km,
        pothole_detected,
        pothole_count,
        crack_intensity,
        texture_depth_variance,
        bump_intensity,
        rut_depth,
        edge_roughness,

        -- Severity Score (0-100)
        -- Weights inspired by the research paper's MRMR feature ranking
        ROUND(
            (texture_depth_variance * 2.5) +   -- top feature from paper
            (bump_intensity         * 2.0) +   -- 2nd feature from paper
            (crack_intensity        * 1.8) +   -- critical for water ingress
            (rut_depth              * 1.5) +   -- leads to water ponding
            (edge_roughness         * 1.2) +   -- edge pothole indicator
            (accidents_per_km       * 1.5) +   -- MoRTH accident weighting
            (CASE road_type
                WHEN 'highway'   THEN 3.0      -- high speed = high danger
                WHEN 'arterial'  THEN 2.0
                WHEN 'collector' THEN 1.0
                ELSE 0.5
             END)
        , 2) AS severity_score,

        -- Priority Label
        CASE
            WHEN (
                (texture_depth_variance * 2.5) +
                (bump_intensity         * 2.0) +
                (crack_intensity        * 1.8) +
                (rut_depth              * 1.5) +
                (edge_roughness         * 1.2) +
                (accidents_per_km       * 1.5) +
                (CASE road_type
                    WHEN 'highway'   THEN 3.0
                    WHEN 'arterial'  THEN 2.0
                    WHEN 'collector' THEN 1.0
                    ELSE 0.5
                 END)
            ) >= 35 THEN 'PRIORITY_1_CRITICAL'
            WHEN (
                (texture_depth_variance * 2.5) +
                (bump_intensity         * 2.0) +
                (crack_intensity        * 1.8) +
                (rut_depth              * 1.5) +
                (edge_roughness         * 1.2) +
                (accidents_per_km       * 1.5) +
                (CASE road_type
                    WHEN 'highway'   THEN 3.0
                    WHEN 'arterial'  THEN 2.0
                    WHEN 'collector' THEN 1.0
                    ELSE 0.5
                 END)
            ) >= 20 THEN 'PRIORITY_2_HIGH'
            ELSE 'PRIORITY_3_MONITOR'
        END AS priority_label

    FROM pothole_heatmap.road_sections_raw
""")

print(f"✅ Scored {scored_df.count():,} road sections")
scored_df.show(5)

✅ Scored 50,000 road sections
+----------+---------+--------+---------+---------+-------------+----------------+----------------+-------------+---------------+----------------------+--------------+---------+--------------+--------------+-------------------+
|section_id|     city|latitude|longitude|road_type|daily_traffic|accidents_per_km|pothole_detected|pothole_count|crack_intensity|texture_depth_variance|bump_intensity|rut_depth|edge_roughness|severity_score|     priority_label|
+----------+---------+--------+---------+---------+-------------+----------------+----------------+-------------+---------------+----------------------+--------------+---------+--------------+--------------+-------------------+
|         0|  Kolkata| 18.8528|   95.129|    local|        62403|            7.19|               1|            1|           0.28|                  2.83|          3.91|     5.45|           2.3|         37.62|PRIORITY_1_CRITICAL|
|         1|   Mumbai| 21.6102|  82.9091|  highway|       

In [0]:
# CELL 2 — Save scored results to Delta Lake
scored_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pothole_heatmap.road_sections_scored")

print("✅ Saved scored table to Delta Lake!")

✅ Saved scored table to Delta Lake!


In [0]:
# CELL 3 — Summary: how many sections in each priority band per city
summary = spark.sql("""
    SELECT
        city,
        priority_label,
        COUNT(*) as section_count,
        ROUND(AVG(severity_score), 1) as avg_severity,
        ROUND(AVG(accidents_per_km), 2) as avg_accidents,
        SUM(pothole_detected) as potholes_detected
    FROM pothole_heatmap.road_sections_scored
    GROUP BY city, priority_label
    ORDER BY city, priority_label
""")

summary.show(30)

+---------+-------------------+-------------+------------+-------------+-----------------+
|     city|     priority_label|section_count|avg_severity|avg_accidents|potholes_detected|
+---------+-------------------+-------------+------------+-------------+-----------------+
|Bengaluru|PRIORITY_1_CRITICAL|         7764|        57.0|         6.16|             3934|
|Bengaluru|    PRIORITY_2_HIGH|          461|        30.5|         3.29|              128|
|Bengaluru| PRIORITY_3_MONITOR|           18|        17.2|         1.49|                4|
|  Chennai|PRIORITY_1_CRITICAL|         7830|        57.5|         6.26|             4043|
|  Chennai|    PRIORITY_2_HIGH|          449|        30.3|          3.5|              133|
|  Chennai| PRIORITY_3_MONITOR|           12|        15.6|         1.78|                0|
|    Delhi|PRIORITY_1_CRITICAL|         7853|        57.5|         6.15|             4006|
|    Delhi|    PRIORITY_2_HIGH|          465|        30.4|         3.22|              115|

In [0]:
# CELL 4 — Find the top 20 most critical road sections (for the demo)
top_critical = spark.sql("""
    SELECT
        section_id,
        city,
        ROUND(latitude, 4)  as lat,
        ROUND(longitude, 4) as lon,
        road_type,
        severity_score,
        priority_label,
        ROUND(accidents_per_km, 1) as accidents_per_km,
        pothole_count
    FROM pothole_heatmap.road_sections_scored
    WHERE priority_label = 'PRIORITY_1_CRITICAL'
    ORDER BY severity_score DESC
    LIMIT 20
""")

top_critical.show()
print("✅ These are your Priority 1 sections — the heatmap will highlight these in RED")

+----------+---------+-------+-------+---------+--------------+-------------------+----------------+-------------+
|section_id|     city|    lat|    lon|road_type|severity_score|     priority_label|accidents_per_km|pothole_count|
+----------+---------+-------+-------+---------+--------------+-------------------+----------------+-------------+
|     34848|   Mumbai|13.8778|87.0387|  highway|        102.73|PRIORITY_1_CRITICAL|            11.7|            4|
|     36606|   Mumbai| 18.566|75.2563|  highway|        101.32|PRIORITY_1_CRITICAL|            11.9|            4|
|      9389|   Mumbai|29.8683|77.4782|    local|         99.36|PRIORITY_1_CRITICAL|             9.6|            4|
|     33756|  Kolkata|23.8326|75.5418|  highway|         98.73|PRIORITY_1_CRITICAL|            10.2|            4|
|      4935|  Kolkata|23.1214|75.2151| arterial|         97.95|PRIORITY_1_CRITICAL|            11.6|            0|
|      9999|   Mumbai|18.7783|82.0296|  highway|         97.33|PRIORITY_1_CRITIC

In [0]:
# CELL 5 — Auto-generate PWD alerts for Priority 1 sections
from pyspark.sql import functions as F

alerts_df = spark.sql("""
    SELECT
        section_id,
        city,
        ROUND(latitude, 4)  as latitude,
        ROUND(longitude, 4) as longitude,
        road_type,
        severity_score,
        ROUND(accidents_per_km, 1) as accidents_per_km,
        pothole_count,
        CONCAT(
            'URGENT NOTICE TO PWD ', UPPER(city), ' DIVISION | ',
            'Road Section ID: ', section_id, ' | ',
            'Type: ', UPPER(road_type), ' | ',
            'Severity Score: ', severity_score, '/100 | ',
            'Accident Rate: ', ROUND(accidents_per_km, 1),
            ' accidents/km/year (MoRTH Data) | ',
            'Potholes Detected: ', pothole_count, ' | ',
            'ACTION REQUIRED: Immediate inspection and repair warranted ',
            'under Section 8 of Motor Vehicles Act 1988.'
        ) as auto_notice
    FROM pothole_heatmap.road_sections_scored
    WHERE priority_label = 'PRIORITY_1_CRITICAL'
    ORDER BY severity_score DESC
    LIMIT 10
""")

alerts_df.select("city", "road_type", "severity_score", "auto_notice").show(10, truncate=80)

# Save alerts table
alerts_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pothole_heatmap.pwd_alerts")

print("✅ PWD alerts table saved!")
print(f"Sample notice:\n{alerts_df.first()['auto_notice']}")

+---------+---------+--------------+--------------------------------------------------------------------------------+
|     city|road_type|severity_score|                                                                     auto_notice|
+---------+---------+--------------+--------------------------------------------------------------------------------+
|   Mumbai|  highway|        102.73|URGENT NOTICE TO PWD MUMBAI DIVISION | Road Section ID: 34848 | Type: HIGHWAY...|
|   Mumbai|  highway|        101.32|URGENT NOTICE TO PWD MUMBAI DIVISION | Road Section ID: 36606 | Type: HIGHWAY...|
|   Mumbai|    local|         99.36|URGENT NOTICE TO PWD MUMBAI DIVISION | Road Section ID: 9389 | Type: LOCAL | ...|
|  Kolkata|  highway|         98.73|URGENT NOTICE TO PWD KOLKATA DIVISION | Road Section ID: 33756 | Type: HIGHWA...|
|  Kolkata| arterial|         97.95|URGENT NOTICE TO PWD KOLKATA DIVISION | Road Section ID: 4935 | Type: ARTERIA...|
|   Mumbai|  highway|         97.33|URGENT NOTICE TO PWD

In [0]:
# CELL 6 — Fixed version
decay_df = spark.sql("""
    SELECT s.*,
        r.survey_year,
        ROUND(
            s.severity_score * (1 + (2025 - r.survey_year) * 0.15)
        , 2) as urgency_score,
        CASE
            WHEN r.survey_year <= 2021 THEN 'Chronic (4+ years)'
            WHEN r.survey_year = 2022  THEN 'Persistent (3 years)'
            WHEN r.survey_year = 2023  THEN 'Developing (2 years)'
            ELSE 'Recent (1 year)'
        END as road_age_status
    FROM pothole_heatmap.road_sections_scored s
    JOIN pothole_heatmap.road_sections_raw r 
    ON s.section_id = r.section_id
""")

decay_df.select(
    "city", "road_type", "survey_year",
    "severity_score", "urgency_score", "road_age_status"
).orderBy("urgency_score", ascending=False).show(10)

+---------+---------+-----------+--------------+-------------+------------------+
|     city|road_type|survey_year|severity_score|urgency_score|   road_age_status|
+---------+---------+-----------+--------------+-------------+------------------+
|   Mumbai|  highway|       2021|        102.73|       164.37|Chronic (4+ years)|
|Bengaluru|    local|       2021|          97.1|       155.36|Chronic (4+ years)|
|   Mumbai| arterial|       2021|         96.01|       153.62|Chronic (4+ years)|
|  Chennai|    local|       2021|         95.33|       152.53|Chronic (4+ years)|
|Bengaluru|collector|       2021|         95.01|       152.02|Chronic (4+ years)|
|  Kolkata|collector|       2021|          94.3|       150.88|Chronic (4+ years)|
|Hyderabad|    local|       2021|          94.1|       150.56|Chronic (4+ years)|
|Bengaluru|  highway|       2021|         94.02|       150.43|Chronic (4+ years)|
|    Delhi|  highway|       2021|         93.97|       150.35|Chronic (4+ years)|
|Hyderabad| arte

In [0]:
# CELL 7 — Save enhanced scores
decay_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pothole_heatmap.road_sections_urgency")

print("✅ Temporal decay scores saved!")
print("   A road damaged in 2021 scores 60% higher than same damage in 2024")

✅ Temporal decay scores saved!
   A road damaged in 2021 scores 60% higher than same damage in 2024
